In [11]:
import os
import mmap
import time
import numpy as np

PATH = "200x.000"

In [12]:
def scan_seek(path):
    file = open(path, "rb")
    # Get file size for progress bar
    file.seek(0, 2)
    file_size = file.tell()
    file.seek(0)

    # List for storing start index of each ensamble
    current_offset = 0
    ens_indexes = []

    # Search the entire file for ensemble headers
    while True:
        # Get header ID, source ID, and numbytes
        header = file.read(4)

        # End of file check
        if len(header) < 4:
            break

        # If the two start bytes are right (two 7f bytes in a row)
        if header[0] == 0x7f and header[1] == 0x7f:
            # Get ensemble size
            ens_size = header[2] + (header[3] << 8) + 2

            # If size is within expected bounds (removes potential errors from random 7f7f data)
            if 32 <= ens_size <= 4096:
                ens_indexes.append(current_offset)
                current_offset += ens_size
                file.seek(current_offset)
            else:
                # Continue seeking
                current_offset += 1
                file.seek(current_offset)
        else:
            # Continue seeking
            current_offset += 1
            file.seek(current_offset)

    return ens_indexes

In [13]:
def scan_mmap(path):
    with open(path, "rb") as f:
        file_size = os.path.getsize(path)
        mm = mmap.mmap(f.fileno(), length=0, access=mmap.ACCESS_READ)

        current_offset = 0
        ens_indexes = []

        while current_offset < file_size:
            if current_offset + 4 > file_size:
                break  # not enough bytes left for a header
        
            header = mm[current_offset:current_offset + 4]
        
            if header[0] == 0x7f and header[1] == 0x7f:
                ens_size = header[2] + (header[3] << 8) + 2
                if 32 <= ens_size <= 4096 and current_offset + ens_size <= file_size:
                    ens_indexes.append(current_offset)
                    current_offset += ens_size
                    continue
        
            current_offset += 1

        mm.close()
        return ens_indexes

In [4]:
%timeit -n 10 scan_seek(PATH)
%timeit -n 10 scan_mmap(PATH)

2.82 s ± 57.1 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)
1.11 s ± 21.3 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [14]:
ens_indexes_seek = scan_seek(PATH)

In [15]:
ens_indexes_mmap = scan_mmap(PATH)

In [16]:
seek_set = set(ens_indexes_seek)
mmap_set = set(ens_indexes_mmap)

missing = list(seek_set - mmap_set)
extra = list(mmap_set - seek_set)

print(f"Missing in mmap: {len(missing)}")
print(f"Extra in mmap: {len(extra)}")

if missing:
    print(f"First missing index: {min(missing)}")

Missing in mmap: 1
Extra in mmap: 0
First missing index: 7104100197


In [17]:
ens_indexes_seek[-10:]

[7104080091,
 7104082325,
 7104084559,
 7104086793,
 7104089027,
 7104091261,
 7104093495,
 7104095729,
 7104097963,
 7104100197]

In [18]:
os.path.getsize(PATH)

7104102400

In [19]:
ens_indexes_mmap[-10:]

[7104077857,
 7104080091,
 7104082325,
 7104084559,
 7104086793,
 7104089027,
 7104091261,
 7104093495,
 7104095729,
 7104097963]